# ComplaintGuard Day 10 — NLLB synthetic development evaluation

This notebook evaluates one pinned Myanmar-to-English candidate on **30 synthetic development cases only**. It is not final frozen-validation evidence, does not tune the frozen classifier, and contains no real complaint data.

Base: `facebook/nllb-200-distilled-600M` at `f8d333a098d19b4fd9a8b18f94170487ad3f821d`  
Adapter: `banyaroo/nllb-200-distilled-600m-mya_Mymr-eng_Latn-lora` at `75a3b55efd4802aa1ef7051577354162926e4085`

Both repositories declare CC-BY-NC-4.0. This noncommercial academic evaluation must attribute both creators/repositories, link the license, and identify modifications. Do not use the artifacts commercially or present unreviewed output as validated translation.

## Run order and recovery

Run cells from top to bottom. Expensive acquisition and loading cells validate reusable state in the same live Colab session. If the session disconnects, restart from the RAM/disk gate; `/content` may have been erased. Interrupted acquisition or evaluation is never reported as complete. Google Drive, GitHub authentication, hosted inference, and paid services are not used.

In [ ]:
# Runtime safety gate — this cell must run before installation or acquisition.
import os, platform, shutil, sys
from pathlib import Path

def mem_total_bytes():
    for line in Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemTotal:'):
            return int(line.split()[1]) * 1024
    raise RuntimeError('runtime_ram_unavailable')

RUNTIME_ROOT = Path('/content/complaintguard_day10')
CACHE_DIR = RUNTIME_ROOT / 'hf_cache'
UPLOAD_DIR = RUNTIME_ROOT / 'uploads'
OUTPUT_DIR = RUNTIME_ROOT / 'outputs'
ram_bytes = mem_total_bytes()
disk = shutil.disk_usage('/content')
print({'python': platform.python_version(), 'platform': platform.platform(),
       'physical_ram_bytes': ram_bytes, 'free_disk_bytes': disk.free})
try:
    import torch
    print({'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(),
           'runtime_type': 'GPU' if torch.cuda.is_available() else 'CPU'})
except ImportError:
    print({'torch': 'not_available', 'cuda_available': False, 'runtime_type': 'CPU'})
if ram_bytes < 12_000_000_000:
    raise RuntimeError('preflight_ram_below_12GB_stop_before_install_or_download')
if disk.free < 6_500_000_000:
    raise RuntimeError('preflight_disk_below_6_5GB_stop_before_install_or_download')
for path in (RUNTIME_ROOT, CACHE_DIR, UPLOAD_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
PREFLIGHT = {'physical_ram_bytes': ram_bytes, 'free_disk_bytes_before': disk.free,
             'runtime_type': 'GPU' if 'torch' in sys.modules and torch.cuda.is_available() else 'CPU'}
print('PRECONDITIONS_PASSED')

## Upload exactly three approved local artifacts

- `myanmar_checkpoint_dev_v1.json` — synthetic development cases
- `cfpb_department_model_v1.joblib` — frozen Day 9 vectorizer/classifier bundle
- `cfpb_model_v1_metrics.json` — aggregate model contract and integrity metadata

Do not upload datasets, caches, validation evidence, credentials, or repository archives.

In [ ]:
import hashlib, importlib.metadata as runtime_metadata, json, re
from collections import Counter
from google.colab import files

EXPECTED_UPLOADS = {
    'myanmar_checkpoint_dev_v1.json': 'd21f05ce31c64ca4e3d9c14f0267b5b542e35bf6c120c9e49ef6dab5339cf2ec',
    'cfpb_department_model_v1.joblib': 'bafc086fe5b11bdcc5cbc4f04f3f3f222de8cbad27fe66d62a6685cc30f953d5',
    'cfpb_model_v1_metrics.json': '99fc40b8e791fe65ff7ed22e8e5a731ed650351ad577d27322e95f2bdd1550d8',
}
LABELS = ('transfer_payment', 'account_support', 'card_atm', 'fraud_security', 'loan_credit', 'general_support')
uploaded = files.upload()
if set(uploaded) != set(EXPECTED_UPLOADS):
    raise RuntimeError('upload_set_must_equal_three_approved_files')
for name, payload in uploaded.items():
    digest = hashlib.sha256(payload).hexdigest()
    if digest != EXPECTED_UPLOADS[name]:
        raise RuntimeError(f'upload_hash_mismatch:{name}')
    (UPLOAD_DIR / name).write_bytes(payload)

dev_cases = json.loads((UPLOAD_DIR / 'myanmar_checkpoint_dev_v1.json').read_text('utf-8'))
required = {'id','myanmar_input','expected_english_intent','expected_department','style_tags','mixed_language_numeric_tags'}
if len(dev_cases) != 30 or len({x['id'] for x in dev_cases}) != 30:
    raise RuntimeError('development_record_count_or_id_failure')
if any(set(x) != required or x['expected_department'] not in LABELS for x in dev_cases):
    raise RuntimeError('development_schema_or_label_failure')
if Counter(x['expected_department'] for x in dev_cases) != Counter({x: 5 for x in LABELS}):
    raise RuntimeError('development_department_distribution_failure')
if any(not x['myanmar_input'].strip() or not x['expected_english_intent'].strip() for x in dev_cases):
    raise RuntimeError('development_empty_text_failure')
metrics_contract = json.loads((UPLOAD_DIR / 'cfpb_model_v1_metrics.json').read_text('utf-8'))
if metrics_contract.get('status') != 'completed' or metrics_contract.get('model_version') != 'v1':
    raise RuntimeError('classifier_metrics_contract_failure')
generated = metrics_contract.get('generated_model', {})
if generated.get('sha256') != EXPECTED_UPLOADS['cfpb_department_model_v1.joblib']:
    raise RuntimeError('classifier_manifest_hash_failure')
recorded_environment = metrics_contract.get('environment', {})
for package, key in (('scikit-learn','scikit_learn'), ('joblib','joblib')):
    if runtime_metadata.version(package) != recorded_environment.get(key):
        raise RuntimeError(f'colab_classifier_runtime_version_mismatch:{package}')
UPLOAD_HASHES = dict(EXPECTED_UPLOADS)
print('UPLOAD_VALIDATION_PASSED', Counter(x['expected_department'] for x in dev_cases))

## Install the approved candidate runtime

This modifies only the temporary Colab runtime. PyTorch comes from Colab. SentencePiece is installed only when absent. The frozen classifier is loaded with Colab-provided scikit-learn/joblib; its serialized contract and hash are checked before use.

In [ ]:
import importlib.util, subprocess
pins = ['transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0']
if importlib.util.find_spec('sentencepiece') is None:
    pins.append('sentencepiece==0.2.1')
else:
    import importlib.metadata as preinstall_metadata
    if not preinstall_metadata.version('sentencepiece').startswith('0.2.'):
        raise RuntimeError('existing_sentencepiece_version_is_not_compatible')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *pins], check=True)
import importlib.metadata as metadata
DEPENDENCIES = {name: metadata.version(name) for name in
                ('torch','transformers','peft','accelerate','sentencepiece','scikit-learn','joblib')}
for name, version in {'transformers':'4.51.3','peft':'0.15.2','accelerate':'1.6.0'}.items():
    if DEPENDENCIES[name] != version:
        raise RuntimeError(f'dependency_pin_failure:{name}')
print(DEPENDENCIES)

## Acquire only the pinned runtime files

The notebook-local cache is measured by unique physical file identity. The acquisition cell records every required runtime file, its byte size and SHA-256, download duration, peak observed cache size, final cache size, and remaining disk. It stops if the final unique cache exceeds 2.6 GB.

In [ ]:
import threading, time
from huggingface_hub import snapshot_download

BASE_ID = 'facebook/nllb-200-distilled-600M'
BASE_REV = 'f8d333a098d19b4fd9a8b18f94170487ad3f821d'
ADAPTER_ID = 'banyaroo/nllb-200-distilled-600m-mya_Mymr-eng_Latn-lora'
ADAPTER_REV = '75a3b55efd4802aa1ef7051577354162926e4085'
BASE_FILES = ['config.json','generation_config.json','pytorch_model.bin','sentencepiece.bpe.model',
              'special_tokens_map.json','tokenizer.json','tokenizer_config.json']
ADAPTER_FILES = ['adapter_config.json','adapter_model.bin']

def unique_physical_size(root):
    seen, total = set(), 0
    for path in Path(root).rglob('*'):
        if path.is_file():
            stat = path.stat()
            key = (stat.st_dev, stat.st_ino)
            if key not in seen:
                seen.add(key); total += stat.st_size
    return total

peak = {'bytes': unique_physical_size(CACHE_DIR), 'running': True}
def monitor():
    while peak['running']:
        peak['bytes'] = max(peak['bytes'], unique_physical_size(CACHE_DIR))
        time.sleep(0.25)

from datetime import datetime, timezone
acquisition_started_utc = datetime.now(timezone.utc).isoformat()
started = time.perf_counter(); watcher = threading.Thread(target=monitor, daemon=True); watcher.start()
try:
    base_snapshot = Path(snapshot_download(repo_id=BASE_ID, revision=BASE_REV, cache_dir=CACHE_DIR,
        allow_patterns=BASE_FILES))
    adapter_snapshot = Path(snapshot_download(repo_id=ADAPTER_ID, revision=ADAPTER_REV, cache_dir=CACHE_DIR,
        allow_patterns=ADAPTER_FILES))
finally:
    peak['running'] = False; watcher.join()
acquisition_seconds = time.perf_counter() - started
acquisition_finished_utc = datetime.now(timezone.utc).isoformat()
peak['bytes'] = max(peak['bytes'], unique_physical_size(CACHE_DIR))

def sha256_path(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def inventory(snapshot, names, prefix):
    rows = []
    for name in names:
        path = snapshot / name
        if not path.is_file(): raise RuntimeError(f'acquired_file_missing:{prefix}/{name}')
        rows.append({'repository': prefix, 'name': name, 'size_bytes': path.stat().st_size,
                     'sha256': sha256_path(path)})
    return rows
ACQUIRED_FILES = inventory(base_snapshot, BASE_FILES, BASE_ID) + inventory(adapter_snapshot, ADAPTER_FILES, ADAPTER_ID)
candidate_cache_bytes = unique_physical_size(CACHE_DIR)
if candidate_cache_bytes > 2_600_000_000:
    raise RuntimeError('candidate_cache_exceeds_2_6GB')
ACQUISITION = {'started_at_utc': acquisition_started_utc, 'finished_at_utc': acquisition_finished_utc,
               'seconds': acquisition_seconds, 'logical_runtime_bytes': sum(x['size_bytes'] for x in ACQUIRED_FILES),
               'unique_physical_cache_bytes': candidate_cache_bytes, 'peak_observed_cache_bytes': peak['bytes'],
               'remaining_disk_bytes': shutil.disk_usage('/content').free}
print(ACQUISITION); print(ACQUIRED_FILES)

## Load and verify the composed candidate offline

The source language is `mya_Mymr`; generation forces `eng_Latn`. Generation is greedy and deterministic. No prompt, keyword rule, translation correction, or routing post-processing is applied.

In [ ]:
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'
import gc, joblib, numpy as np, torch, unicodedata
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GENERATION = {'max_new_tokens': 256, 'do_sample': False, 'num_beams': 1}
def load_candidate():
    tokenizer = AutoTokenizer.from_pretrained(base_snapshot, src_lang='mya_Mymr', tgt_lang='eng_Latn', local_files_only=True)
    base = AutoModelForSeq2SeqLM.from_pretrained(base_snapshot, local_files_only=True, use_safetensors=False)
    model = PeftModel.from_pretrained(base, adapter_snapshot, local_files_only=True).to(DEVICE).eval()
    forced = tokenizer.convert_tokens_to_ids('eng_Latn')
    if forced in (None, tokenizer.unk_token_id): raise RuntimeError('english_bos_token_missing')
    return tokenizer, model, forced

if not globals().get('CANDIDATE_LOADED', False):
    load_started = time.perf_counter(); first_tokenizer, first_model, first_bos = load_candidate()
    cold_load_seconds = time.perf_counter() - load_started
    # Release the first construction before a second strict-offline construction.
    del first_tokenizer, first_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    tokenizer, translation_model, forced_bos = load_candidate()
    CANDIDATE_LOADED = True
else:
    first_bos = forced_bos
    cold_load_seconds = OFFLINE_RELOAD['cold_load_seconds']
OFFLINE_RELOAD = {'passed': first_bos == forced_bos, 'cold_load_seconds': cold_load_seconds,
                  'device': str(DEVICE), 'forced_bos_token_id': forced_bos}

artifact = joblib.load(UPLOAD_DIR / 'cfpb_department_model_v1.joblib')
if tuple(artifact.get('labels', ())) != LABELS or artifact.get('fallback_label') != 'general_support':
    raise RuntimeError('frozen_classifier_contract_failure')
if artifact.get('model_version') != 'v1' or artifact.get('dataset_version') != 'v1' or artifact.get('mapping_version') != 'v1':
    raise RuntimeError('frozen_classifier_version_failure')

def normalize_english(text):
    return ' '.join(unicodedata.normalize('NFKC', text).casefold().split())
def classify(text):
    matrix = artifact['vectorizer'].transform([normalize_english(text)])
    probabilities = artifact['classifier'].predict_proba(matrix)[0]
    idx = int(np.argmax(probabilities)); label = str(artifact['classifier'].classes_[idx]); confidence = float(probabilities[idx])
    fallback = confidence < float(artifact['confidence_threshold'])
    return {'department': 'general_support' if fallback else label, 'confidence': confidence, 'fallback': fallback}
print('STRICT_OFFLINE_RELOAD_PASSED', OFFLINE_RELOAD)

## Run development-only translation and classification

Outputs are synthetic development evidence. Semantic translation quality remains `pending_owner_review`; this notebook records execution and mechanical checks but never fabricates human scores.

In [ ]:
import resource
def translate(text):
    encoded = tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    with torch.inference_mode():
        generated = translation_model.generate(**encoded, forced_bos_token_id=forced_bos, **GENERATION)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()
def mechanical_flags(text):
    words = text.split()
    repetition = len(words) >= 6 and len(set(words)) / len(words) < 0.35
    unexpected_myanmar = bool(re.search(r'[\u1000-\u109f\ua9e0-\ua9ff\uaa60-\uaa7f]', text))
    return {'empty_output': not bool(text), 'obvious_repetition': repetition, 'unexpected_myanmar_script': unexpected_myanmar}

run_started = time.perf_counter(); results = []
for case in dev_cases:
    item_started = time.perf_counter()
    try:
        english = translate(case['myanmar_input'])
        if not english: raise RuntimeError('empty_translation')
        prediction = classify(english)
        status, error = 'success', None
    except Exception as exc:
        english = ''; prediction = {'department': None, 'confidence': None, 'fallback': None}
        status, error = 'error', {'code': type(exc).__name__}
    results.append({'development_id': case['id'], 'myanmar_input': case['myanmar_input'],
        'generated_english_translation': english, 'expected_english_intent': case['expected_english_intent'],
        'expected_department': case['expected_department'], 'classifier_prediction': prediction['department'],
        'classifier_confidence': prediction['confidence'],
        'classification_correct': prediction['department'] == case['expected_department'],
        'translation_execution_status': status, 'inference_seconds': time.perf_counter()-item_started,
        'error': error, 'mechanical_checks': mechanical_flags(english),
        'semantic_translation_review': 'pending_owner_review'})
total_runtime = time.perf_counter() - run_started
# Repeat three development-only inputs without modifying results.
repeat_ids = [dev_cases[i]['id'] for i in (0, 10, 20)]
repeatability = []
for case_id in repeat_ids:
    case = next(x for x in dev_cases if x['id'] == case_id)
    original = next(x['generated_english_translation'] for x in results if x['development_id'] == case_id)
    repeated = translate(case['myanmar_input'])
    repeatability.append({'development_id': case_id, 'identical': original == repeated})
peak_rss_bytes = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024)
print('DEVELOPMENT_EXECUTION_FINISHED', {'cases': len(results), 'seconds': total_runtime})

In [ ]:
from datetime import datetime, timezone
successes = sum(x['translation_execution_status'] == 'success' for x in results)
correct = sum(x['classification_correct'] for x in results)
by_department = {}
for label in LABELS:
    rows = [x for x in results if x['expected_department'] == label]
    by_department[label] = {'total': len(rows), 'correct': sum(x['classification_correct'] for x in rows)}
summary = {'evidence_type': 'synthetic_development_only', 'semantic_review_status': 'pending_owner_review',
    'total_cases': len(results), 'successful_translations': successes, 'empty_or_error_outputs': len(results)-successes,
    'classification_correct': correct, 'classification_by_department': by_department,
    'total_runtime_seconds': total_runtime, 'mean_inference_seconds': sum(x['inference_seconds'] for x in results)/len(results),
    'peak_process_rss_bytes': peak_rss_bytes, 'deterministic_repeatability': repeatability,
    'mechanical_flag_counts': {key: sum(x['mechanical_checks'][key] for x in results)
        for key in ('empty_output','obvious_repetition','unexpected_myanmar_script')}}
results_doc = {'status': 'completed', 'evidence_type': 'synthetic_development_only', 'results': results}
results_path = OUTPUT_DIR / 'myanmar_nllb_dev_results.json'
summary_path = OUTPUT_DIR / 'myanmar_nllb_dev_summary.json'
results_path.write_text(json.dumps(results_doc, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
output_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (results_path, summary_path)}
manifest = {'status': 'completed', 'evidence_type': 'synthetic_development_only',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'base': {'id': BASE_ID, 'revision': BASE_REV}, 'adapter': {'id': ADAPTER_ID, 'revision': ADAPTER_REV},
    'dependencies': DEPENDENCIES, 'runtime_preflight': PREFLIGHT, 'generation': GENERATION,
    'input_artifact_hashes': UPLOAD_HASHES, 'acquired_files': ACQUIRED_FILES,
    'acquisition': ACQUISITION, 'offline_reload': OFFLINE_RELOAD,
    'output_artifact_hashes': output_hashes, 'semantic_review_status': 'pending_owner_review'}
manifest_path = OUTPUT_DIR / 'myanmar_nllb_colab_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
print(json.dumps(summary, indent=2)); print('MANIFEST_SHA256', hashlib.sha256(manifest_path.read_bytes()).hexdigest())

## Export and owner-review boundary

Download all three JSON files and return them for repository-side validation. Do not run any other case set. The generated translations still require owner human review before candidate acceptance or any separately authorized frozen-validation run.

In [ ]:
for path in (results_path, summary_path, manifest_path):
    if not path.is_file(): raise RuntimeError(f'completed_output_missing:{path.name}')
    files.download(str(path))